# VisTAD: VLM-Guided Distillation for Time-Series Anomaly Detection
### Complete Kaggle Pipeline: Setup, Caching, Training & Continuous Evaluation

This notebook guides you step-by-step through running the **VisTAD** architecture on Kaggle.

#### Core Concept
1. **Teacher (Vision-Language Model - Qwen2.5-VL-3B)**: Analyzes dual-horizon telemetry line charts (macro context + micro inspection window), filtering down to the Top-6 active channels to avoid sensory overload.
2. **Offline Guidance Caching**: Runs the heavy 3B parameter VLM offline once to extract calibrated anomaly scores and 2048-dim latent features.
3. **Student Distillation (1D-CNN / BiLSTM / TimesNet)**: Trains a sub-millisecond student model using a multi-objective loss:
   $$\mathcal{L}_{\text{total}} = w_{\text{sup}} \mathcal{L}_{\text{sup}} + w_{\text{feat}} \mathcal{L}_{\text{feat}} + w_{\text{resp}} \mathcal{L}_{\text{resp}}$$
4. **Continuous Streaming Evaluation**: Evaluates on real continuous streams using Peaks-Over-Threshold (POT) dynamic thresholding and range-aware Affiliation-F1.

---
> **Kaggle Notebook Settings Requirement:**
> - **Accelerator:** GPU T4 x2 (recommended for dual-GPU data parallelism) or GPU P100.
> - **Internet:** ON (to install packages and download benchmarks).

In [ ]:
# --- Step 1: Environment & GPU Hardware Check ---
import torch

print(f"PyTorch Version: {torch.__version__}")
has_cuda = torch.cuda.is_available()
print(f"CUDA Available:  {has_cuda}")

if has_cuda:
    device_count = torch.cuda.device_count()
    print(f"Found {device_count} GPU device(s):")
    for i in range(device_count):
        props = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: CUDA is not available. Please enable GPU under Kaggle Settings -> Accelerator.")

!nvidia-smi

## Step 2: Clone VisTAD Repository & Install Dependencies

We ensure we are inside the repository workspace and install the required dependencies:
- `transformers >= 4.49.0`
- `qwen-vl-utils >= 0.0.8`
- `accelerate >= 0.26.0`
- `scikit-learn`, `scipy`, `pandas`, `pyyaml`, `matplotlib`

In [ ]:
# --- Step 2: Clone Repo & Install Packages ---
import os
import sys

# Clone repository if running in a fresh Kaggle session
if not os.path.exists("VisTAD"):
    !git clone https://github.com/rhythm-kedia/VisTAD.git
    %cd VisTAD
elif os.path.exists("run_teacher_cache.py"):
    print("Already inside VisTAD workspace.")
else:
    %cd VisTAD

# Install requirements quietly
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")

## Step 3: Connect Attached Kaggle Dataset & Caches

Assume the dataset archive (`vistad-datasets` or `data_repository.zip`) is attached to this notebook under `/kaggle/input/`.
The cell below auto-detects and mounts the benchmarks (`smd`, `swat`, `msl`, `gecco`) and loads pre-computed teacher caches into local `cache/`.

In [ ]:
# --- Step 3: Mount Benchmarks & Pre-computed Caches ---
import os, glob, shutil

os.makedirs("data", exist_ok=True)
os.makedirs("cache", exist_ok=True)

for root, dirs, files in os.walk("/kaggle/input"):
    for ds in ["smd", "swat", "msl", "gecco"]:
        if ds in dirs and not os.path.exists(f"data/{ds}"):
            src_p = os.path.join(root, ds)
            dst_p = f"data/{ds}"
            try:
                os.symlink(src_p, dst_p)
            except Exception:
                shutil.copytree(src_p, dst_p)
                
    if "cache" in dirs:
        for pt in glob.glob(os.path.join(root, "cache", "*.pt")):
            dst = os.path.join("cache", os.path.basename(pt))
            if not os.path.exists(dst):
                shutil.copy2(pt, dst)

print(f"Connected benchmarks in data/: {os.listdir('data')}")
print(f"Teacher caches ready in cache/: {len(glob.glob('cache/*.pt'))} file(s)")


## Step 4: Telemetry Exploration & Dual-Horizon Inspection

Before running models, let's inspect the telemetry stream:
- Check channel count and nominal baseline statistics.
- Test **Top-$K$ Channel Filtering**: Extracts only the top 6 most dynamic channels using robust median-shift, preventing sensory clutter.
- Render the **Dual-Horizon Plot**:
  - **Panel 1 (Macro Context, 800 timesteps)**: Establishes baseline nominal system behavior.
  - **Panel 2 (Micro Inspection Window, 200 timesteps)**: Target window under SRE inspection.

In [ ]:
# --- Step 4: Telemetry Inspection & Dual-Horizon Preview ---
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import load_dataset_arrays, select_top_k_channels
from src.teacher_vlm import render_dual_horizon_image

raw_data, labels = load_dataset_arrays(dataset_name="smd", entity="machine-1-1", data_dir="data")

print(f"Dataset Shape:    {raw_data.shape} ({raw_data.shape[1]} channels, {len(raw_data)} timestamps)")
print(f"Anomaly Ratio:    {np.mean(labels):.2%}")

# Select an anomaly window to inspect
anom_indices = np.where(labels == 1)[0]
sample_start = int(anom_indices[0] - 50) if len(anom_indices) > 0 else 3000
sample_end = sample_start + 200

# Base statistics
base_mean = np.mean(raw_data[:3000], axis=0)
base_std = np.std(raw_data[:3000], axis=0) + 1e-6

top_dims = select_top_k_channels(
    raw_data[sample_start:sample_end],
    base_mean=base_mean,
    base_std=base_std,
    strategy="median_shift",
    k=6
)
print(f"Top-6 Active Channels: {top_dims}")

# Render Dual-Horizon Image
img = render_dual_horizon_image(
    raw_data, sample_start, sample_end, top_dims,
    base_mean=base_mean, base_std=base_std, ctx_len=800
)

plt.figure(figsize=(12, 6))
plt.imshow(img)
plt.axis("off")
plt.title(f"Dual-Horizon Telemetry Window @ Timestep {sample_start}:{sample_end} (Top-6 Channels)")
plt.show()

## Step 5: Offline Teacher Caching (`run_teacher_cache.py`)

### Why Offline Caching?
- The Teacher is a 3-Billion parameter Vision-Language Model (`Qwen/Qwen2.5-VL-3B-Instruct`).
- Querying it online during training takes ~8.5 seconds per window.
- By caching balanced curriculum windows (50% anomaly, 50% nominal) offline, the student trains at GPU speed in milliseconds.

### Speedup Optimizations in VisTAD:
1. **Token Budget Capping**: Caps visual tokens to ~252 per image (down from 1,008 default), yielding a **4x compute speedup**.
2. **Dual-GPU Data Parallelism**: Automatically detects 2 GPUs on Kaggle (T4 x2) and shards batches across `cuda:0` and `cuda:1`.
3. **Calibrated Thresholding**: Extracts calibrated logits and 2048-dimensional pooled visual-language feature vectors.

> **Execution Modes:**
> - **Mock Mode (`--mock`)**: Generates simulated teacher guidance in 2 seconds (ideal for dry-run verification).
> - **Full Mode**: Loads `Qwen2.5-VL-3B` on GPU and runs real inference.

In [ ]:
# --- Step 5A: Quick Pipeline Verification (MOCK Mode) ---
# Runs in 3 seconds to verify the pipeline end-to-end
!python run_teacher_cache.py --entity machine-1-1 --num_windows 40 --mock

In [ ]:
# --- Step 5B: Full VLM Teacher Inference (Qwen2.5-VL-3B on GPU) ---
# Run teacher caching for any benchmark of choice:

# 1. SMD (single machine or loop across all 28):
# !python run_teacher_cache.py --dataset smd --entity machine-1-1 --num_windows 40 --batch_size 16

# 2. SWaT (Cyber-physical water treatment):
# !python run_teacher_cache.py --dataset swat --entity swat_attack --num_windows 80 --batch_size 16

# 3. MSL (NASA Mars Science Laboratory):
# !python run_teacher_cache.py --dataset msl --entity msl_test --num_windows 80 --batch_size 16

# 4. GECCO (Drinking Water Quality):
# !python run_teacher_cache.py --dataset gecco --entity gecco_water --num_windows 80 --batch_size 16


In [ ]:
# --- Step 5C: Load Pre-computed Teacher Cache (from attached Kaggle Dataset) ---
import os, glob, shutil, zipfile

os.makedirs("cache", exist_ok=True)

# 1. Auto-extract from any attached cache zip in /kaggle/input or workspace
for z_path in glob.glob("/kaggle/input/**/teacher_cache*.zip", recursive=True) + glob.glob("teacher_cache*.zip"):
    print(f"Extracting pre-computed teacher cache: {z_path} -> cache/...")
    with zipfile.ZipFile(z_path, "r") as z:
        z.extractall("cache")

# 2. Copy loose or dataset-embedded .pt files from /kaggle/input
for pt in glob.glob("/kaggle/input/**/*teacher_cache.pt", recursive=True):
    dst = os.path.join("cache", os.path.basename(pt))
    if not os.path.exists(dst):
        shutil.copy2(pt, dst)

cached_files = glob.glob("cache/*_teacher_cache.pt")
print(f"Cache status: {len(cached_files)} series cache file(s) loaded into local writeable cache/.")


In [ ]:
# --- Step 5D: Package & Export Updated Teacher Cache ---
# When new teacher caches are generated in this run (or to bundle all existing ones),
# this packages everything in cache/ into an exportable zip archive in /kaggle/working/.

import os, glob, zipfile

cache_files = sorted(glob.glob("cache/*_teacher_cache.pt"))
print(f"Total teacher cache files available: {len(cache_files)}")

output_zip = "updated_teacher_cache.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for pt in cache_files:
        z.write(pt, arcname=os.path.basename(pt))

zip_size_mb = os.path.getsize(output_zip) / (1024 * 1024)
print(f"Successfully created '{output_zip}' ({zip_size_mb:.1f} MB) with {len(cache_files)} cache files.")
print("Download this file from Kaggle 'Output' pane to update your Kaggle Dataset!")


## Step 6: Train Student Model with Distillation (`train.py`)

Now we distill the teacher's knowledge into our lightweight **1D-CNN** student.

### Multi-Objective Distillation Loss:
$$\mathcal{L} = w_{\text{sup}} \mathcal{L}_{\text{sup}} + w_{\text{feat}} \mathcal{L}_{\text{feat}} + w_{\text{resp}} \mathcal{L}_{\text{resp}}$$
- $\mathcal{L}_{\text{sup}}$: Imbalance-weighted binary cross-entropy on continuous density and window anomaly flags.
- $\mathcal{L}_{\text{feat}}$: Bounded Cosine distance alignment between student features and teacher 2048-dim latent space.
- $\mathcal{L}_{\text{resp}}$: Soft response matching with threshold-calibrated teacher probabilities.

### Data Split Integrity:
- Chronological split: **50% Train**, **10% Validation**, **40% Test**.
- Window start indices and nominal baselines ($\mu_0, \sigma_0$) are strictly preserved across splits to prevent temporal leakage.

In [ ]:
# --- Step 6: Train 1D-CNN Student ---
# Trains pooled across all cached entities with Cosine Annealing learning rate schedule
!python train.py training.epochs=20 training.batch_size=128 training.lr=0.001

## Step 7: Continuous Stream Evaluation (`evaluate.py`)

Unlike window-level classification, real-world monitoring is a continuous temporal stream:
1. **Continuous Sliding Window ($S=1$ or $S=5$)**: Averages overlapping predictions into a continuous 1D anomaly score curve.
2. **Dynamic Peaks-Over-Threshold (POT)**: Uses Extreme Value Theory (EVT) to compute a dynamic threshold, eliminating manual threshold tuning.
3. **Affiliation-F1**: Evaluates true temporal overlap and event coverage, completely eliminating Point-Adjustment (PA) inflation tricks.

In [ ]:
# --- Step 7: Evaluate Best Checkpoint on Continuous Stream ---
import glob

ckpt_files = glob.glob("checkpoints/*best.pt")
print(f"Available Checkpoints: {ckpt_files}")

best_ckpt = ckpt_files[0] if ckpt_files else "checkpoints/smd_pooled28_cnn_1d_best.pt"
print(f"Evaluating checkpoint: {best_ckpt}")

# Continuous evaluation on representative entities
!python evaluate.py --entity machine-2-8 --checkpoint "{best_ckpt}" --stride 5
!python evaluate.py --entity machine-1-5 --checkpoint "{best_ckpt}" --stride 5

## Step 7B: Head-to-Head Fair Comparison (Teacher vs. Student)

To ensure a 100% apples-to-apples comparison, we evaluate both the **Teacher (Qwen2.5-VL-3B)** and the **distilled Student (1D-CNN)** on the **exact same held-out test windows** ($t \ge 0.60 N$).

In [ ]:
# --- Step 7B: Head-to-Head Fair Comparison ---
!python scripts/compare_teacher_student.py --checkpoint "{best_ckpt}"

## Step 8: Ablation Studies (Backbones & Loss Components)

VisTAD allows hot-swapping model architectures and loss weights directly via command line arguments without modifying code.

### 1. Student Backbone Comparison:
- **1D-CNN** (Proposed default: fast dilated convolutions)
- **BiLSTM** (Recurrent sequence baseline)
- **TimesNet** (Multi-periodic 2D convolutional baseline)

### 2. Distillation Loss Ablations:
- Disable teacher feature alignment: `training.loss_weights.w_feat=0.0`
- Disable teacher response matching: `training.loss_weights.w_resp=0.0`

In [ ]:
# --- Step 8A: Train BiLSTM Student Baseline ---
!python train.py student.architecture=bilstm training.epochs=15 training.batch_size=64

In [ ]:
# --- Step 8B: Distillation Ablation (Supervised Only, No Teacher Guidance) ---
!python train.py training.loss_weights.w_feat=0.0 training.loss_weights.w_resp=0.0 training.epochs=15